In [ ]:
"""
F1 Dataset Preprocessing Pipeline
====================================
Loads:   f1_complete_dataset_2020_2024.csv
Outputs: preprocessed pandas DataFrame  (always)
         f1_complete_preprocessed_dataset_2020_2024.csv  (if SAVE_CSV = True)

Key fix in this version
-----------------------
track_status was being one-hot encoded into 34 columns because FastF1
concatenates status codes when multiple flags are active simultaneously
across different marshal sectors (e.g. "412" = safety car + yellow + clear
in different parts of the circuit). This is meaningless as a category.

Fix: decompose track_status into 4 clean binary flags by checking whether
each important digit is PRESENT anywhere in the string:

    sc_active       : "4" in track_status  (Safety Car deployed)
    vsc_active      : "5" in track_status  (Virtual Safety Car)
    yellow_active   : "2" in track_status  (Yellow flag in any sector)
    red_flag_active : "6" in track_status  (Red flag / session suspended)

This also fixes safety_car_pit in the builder which only checked for
exact matches "4" and "6", missing all concatenated variants.

OHE scope fix
-------------
The dataset was already one-hot encoded by preprocess.py in a previous
run. The CSV loaded here already has columns like current_compound_HARD,
driver_VER, team_Mercedes etc. We must NOT re-encode those.

We only need to OHE columns that are still raw strings, which after the
track_status decomposition is: nothing. All other categoricals were
already encoded. The pipeline detects this automatically.
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

INPUT_FILE  = "f1_complete_dataset_2020_2024.csv"
OUTPUT_FILE = "f1_complete_preprocessed_dataset_2020_2024.csv"

# Set True to save the preprocessed DataFrame as a CSV
SAVE_CSV = True

# KNN imputer neighbours (paper used k=5)
KNN_K = 5

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — LOAD
# ─────────────────────────────────────────────────────────────────────────────

def load_dataset(path: str) -> pd.DataFrame:
    print(f"[1/8] Loading dataset from '{path}'...")
    df = pd.read_csv(path, low_memory=False)
    print(f"      Loaded {len(df):,} rows x {len(df.columns)} columns.")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — DROP COLUMNS
# ─────────────────────────────────────────────────────────────────────────────

DROP_COLUMNS = {

    # --- Explicitly requested drops ---
    "sector1_time"            : "Requested drop; high missingness, redundant with lap_time.",
    "sector2_time"            : "Requested drop; high missingness, redundant with lap_time.",
    "sector3_time"            : "Requested drop; high missingness, redundant with lap_time.",
    "personal_best"           : "Requested drop; binary, rare, low signal-to-noise.",

    # --- Broken computation in dataset_builder.py ---
    # gap_to_car_ahead = lap["Time"] - same_lap["Time"].min()
    #   -> this computes gap to the LEADER, not to the car directly ahead.
    # gap_to_leader = lap["Time"]
    #   -> this is just the session elapsed timestamp, not a gap.
    # Both are wrong. Drop until dataset_builder.py is fixed.
    "gap_to_car_ahead"        : "BROKEN in builder: gives gap-to-leader, not gap to car ahead.",
    "gap_to_leader"           : "BROKEN in builder: raw session timestamp, not a usable gap.",

    # --- Target leakage ---
    "position_change"         : "Leakage: position changes ARE caused by pit stops, not predictive.",

    # --- Static per race, zero lap-level signal ---
    "grid_position"           : "Static per race; live 'position' already captures track position.",
    "qualifying_position"     : "Static per race; also mis-mapped to finish position in builder.",

    # --- Paper explicitly dropped (empirically low significance) ---
    "air_temperature"         : "Paper dropped: low significance for pit stop prediction.",
    "humidity"                : "Paper dropped: low significance for pit stop prediction.",
    "track_temperature"       : "Paper dropped: grouped with AirTemp/Humidity by the paper.",

    # --- Redundant with laps_since_last_pit ---
    "last_pit_lap"            : "Redundant: laps_since_last_pit is the relative version of this.",

    # --- Unreliable hardcoded proxy ---
    "tire_life_remaining_est" : "Unreliable: based on hardcoded TIRE_LIFE_ESTIMATE dict in builder.",

    # --- track_status: dropped here, replaced by 4 clean binary flags in Step 3 ---
    # The raw track_status string from FastF1 is a CONCATENATION of FIA status
    # codes active across different marshal sectors simultaneously.
    # e.g. "412" means Safety Car active (4) + Yellow flag (1→ actually clear,
    # "2" = yellow) + ... it is NOT a meaningful single category.
    # We decompose it into binary flags instead (see Step 3).
    "track_status"            : "Decomposed into sc_active/vsc_active/yellow_active/red_flag_active.",

    # --- safety_car_pit: also replaced ---
    # Builder computed: int(lap["TrackStatus"] in ["4", "6"])
    # This only catches exact matches "4" or "6" and misses ALL concatenated
    # variants like "41", "412", "64", "164", etc.
    # The new sc_active flag from Step 3 is correct and supersedes this.
    "safety_car_pit"          : "Replaced by sc_active (correctly handles concatenated status codes).",
}


def drop_columns(df: pd.DataFrame) -> pd.DataFrame:
    print("\n[2/8] Dropping columns...")
    actually_present = [c for c in DROP_COLUMNS if c in df.columns]
    not_found        = [c for c in DROP_COLUMNS if c not in df.columns]

    for col in actually_present:
        print(f"      DROP  '{col}'")
        print(f"             {DROP_COLUMNS[col]}")
    for col in not_found:
        print(f"      SKIP  '{col}' — not in dataset (already absent or renamed).")

    df = df.drop(columns=actually_present)
    print(f"\n      Columns remaining: {len(df.columns)}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — DECOMPOSE track_status INTO CLEAN BINARY FLAGS
# ─────────────────────────────────────────────────────────────────────────────

def decompose_track_status(df: pd.DataFrame) -> pd.DataFrame:
    """
    FastF1 TrackStatus explained
    ─────────────────────────────
    The FIA timing system broadcasts a status code per marshal sector.
    FastF1 concatenates these into a single string for the lap.

    Individual digit meanings:
        "1"  → Sector clear (normal racing)
        "2"  → Yellow flag  (danger, no overtaking, slow down)
        "4"  → Safety Car deployed
        "5"  → Virtual Safety Car (VSC) — drivers must slow by ~40%
        "6"  → Red flag — session suspended, cars must pit
        "7"  → (reserved / chequered flag sector in some versions)

    Because multiple sectors can have different statuses at the same
    moment, FastF1 concatenates them:
        "41"   → SC active in one sector, clear in another
        "412"  → SC + clear + yellow across different sectors
        "1264" → four different statuses active simultaneously

    This makes the raw string useless as a categorical feature since
    there are dozens of unique combinations (your dataset had 34).

    Solution: check whether each important digit is PRESENT anywhere
    in the string. This correctly handles all concatenation variants.

        sc_active       = "4" in status_string
        vsc_active      = "5" in status_string
        yellow_active   = "2" in status_string
        red_flag_active = "6" in status_string

    Strategic interpretation for the model:
        sc_active / vsc_active  → pit stop is cheap (all cars slow)
                                   teams actively pit under these
        yellow_active           → caution period, partial pit opportunity
        red_flag_active         → race suspended, mandatory pit window
        all flags = 0           → normal green flag racing
    """
    print("\n[3/8] Decomposing track_status into binary flags...")

    # track_status may have already been dropped in Step 2 if it existed
    # as the raw column. Check both raw name and already-encoded variants.
    raw_col = "track_status"

    if raw_col not in df.columns:
        # Check if dataset was loaded from a previously preprocessed CSV
        # that already has track_status_X.X columns — drop those too,
        # they are the broken OHE columns from the old pipeline.
        broken_ts_cols = [c for c in df.columns if c.startswith("track_status_")]
        if broken_ts_cols:
            print(f"      Found {len(broken_ts_cols)} broken OHE track_status columns "
                  f"from a previous run — dropping them.")
            df = df.drop(columns=broken_ts_cols)
        else:
            print("      'track_status' column not found — skipping decomposition.")
            print("      Ensure you are loading the RAW dataset, not a pre-processed one.")
        return df

    # Convert to string safely (handles NaN, float, int variants)
    ts = df[raw_col].fillna("1").astype(str).str.strip()

    df["sc_active"]       = ts.apply(lambda s: int("4" in s))
    df["vsc_active"]      = ts.apply(lambda s: int("5" in s))
    df["yellow_active"]   = ts.apply(lambda s: int("2" in s))
    df["red_flag_active"] = ts.apply(lambda s: int("6" in s))

    # Drop the raw column — it's been fully decomposed
    df = df.drop(columns=[raw_col])

    # Summary
    print(f"      sc_active=1       : {df['sc_active'].sum():,} laps")
    print(f"      vsc_active=1      : {df['vsc_active'].sum():,} laps")
    print(f"      yellow_active=1   : {df['yellow_active'].sum():,} laps")
    print(f"      red_flag_active=1 : {df['red_flag_active'].sum():,} laps")
    print(f"      All clear (0,0,0,0): "
          f"{((df['sc_active']==0)&(df['vsc_active']==0)&(df['yellow_active']==0)&(df['red_flag_active']==0)).sum():,} laps")

    return df


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — FILTER INVALID COMPOUNDS
# ─────────────────────────────────────────────────────────────────────────────

VALID_COMPOUNDS = {"HARD", "MEDIUM", "SOFT"}

# After OHE the compound columns look like current_compound_HARD etc.
# Detect which format is present and filter accordingly.

def filter_compounds(df: pd.DataFrame) -> pd.DataFrame:
    """
    Paper: dropped laps with INTERMEDIATE, WET, UNKNOWN compounds.
    Handles both raw 'current_compound' string column and the already
    OHE'd variant (current_compound_HARD / MEDIUM / SOFT columns).
    """
    print("\n[4/8] Filtering invalid tire compounds...")
    before = len(df)

    if "current_compound" in df.columns:
        # Raw string column present — filter directly
        df = df[df["current_compound"].isin(VALID_COMPOUNDS)].copy()

    elif all(f"current_compound_{c}" in df.columns for c in VALID_COMPOUNDS):
        # Already OHE'd — keep only rows where exactly one of the three
        # valid compound flags is 1 (drops wet/intermediate rows which
        # would have all three = 0)
        valid_mask = (
            df["current_compound_HARD"] |
            df["current_compound_MEDIUM"] |
            df["current_compound_SOFT"]
        ).astype(bool)
        df = df[valid_mask].copy()

    else:
        print("      WARNING: could not identify compound column — skipping filter.")

    after = len(df)
    print(f"      Dropped {before - after:,} rows | Remaining: {after:,}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — SORT & VALIDATE SEQUENTIAL INTEGRITY
# ─────────────────────────────────────────────────────────────────────────────

def sort_and_validate(df: pd.DataFrame) -> pd.DataFrame:
    """
    Paper: 'the data was sorted to ensure sequential ordering before
    proceeding with model building.'
    Sequential integrity is critical — if rows are shuffled, the Bi-LSTM
    will build sequences from non-consecutive laps and learn garbage.
    """
    print("\n[5/8] Sorting for sequential integrity...")

    # Determine driver column name (raw or one of the OHE columns)
    sort_cols = ["season", "round_number", "lap_number"]
    if "driver" in df.columns:
        sort_cols = ["season", "round_number", "driver", "lap_number"]

    df = df.sort_values(sort_cols).reset_index(drop=True)

    # Duplicate check
    dedup_cols = [c for c in ["season", "round_number", "driver", "lap_number"]
                  if c in df.columns]
    if dedup_cols:
        dupes = df.duplicated(subset=dedup_cols).sum()
        if dupes > 0:
            print(f"      WARNING: {dupes:,} duplicate rows — dropping.")
            df = df.drop_duplicates(subset=dedup_cols, keep="first")
        else:
            print("      No duplicate rows. OK")

    return df


# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 — MISSING VALUE HANDLING
# ─────────────────────────────────────────────────────────────────────────────

def handle_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Three-tier imputation strategy:

    Tier 1 — Categorical strings   : mode imputation
    Tier 2 — Binary flags (0/1)    : fill with 0 (event did not occur)
    Tier 3 — Continuous numerics   : StandardScaler + KNN (k=5)
              Scale before KNN because KNN is distance-based. Without
              scaling, lap_time (~70-90s) would dominate position (1-20).

    IMPORTANT for model training later:
    The scaler here is fit on the full dataset for preprocessing purposes.
    When you do your train/test split for the model, refit the scaler
    on training data only and transform the test set with that scaler.
    """
    print("\n[6/8] Handling missing values...")

    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if missing.empty:
        print("      No missing values found. OK")
    else:
        print("      Missing value report:")
        for col, cnt in missing.items():
            print(f"        {col}: {cnt:,} ({cnt/len(df)*100:.2f}%)")

    # Tier 1: categorical string columns
    cat_cols = [c for c in ["current_compound", "driver", "team", "race_name"]
                if c in df.columns and df[c].isnull().any()]
    for col in cat_cols:
        mode = df[col].mode(dropna=True)
        fill = mode.iloc[0] if not mode.empty else "UNKNOWN"
        n    = df[col].isnull().sum()
        df[col].fillna(fill, inplace=True)
        print(f"      Mode-filled '{col}': {n} nulls -> '{fill}'")

    # Tier 2: binary flags — absence of event = 0
    binary_cols = [
        c for c in [
            "is_new_tire", "is_being_attacked", "is_stuck_in_train",
            "sc_active", "vsc_active", "yellow_active", "red_flag_active",
        ]
        if c in df.columns and df[c].isnull().any()
    ]
    for col in binary_cols:
        n = df[col].isnull().sum()
        df[col].fillna(0, inplace=True)
        print(f"      Zero-filled '{col}': {n} nulls")

    # Tier 3: continuous columns — StandardScaler + KNN
    continuous_candidates = [
        "lap_time", "lap_time_delta_prev", "tire_age_laps",
        "laps_since_last_pit", "avg_lap_time_on_stint", "avg_pit_time_team",
        "cars_within_2s_ahead", "cars_within_2s_behind",
        "position", "total_pit_stops_so_far", "stint_number", "lap_number",
        "race_progress_fraction",
    ]
    continuous_cols = [
        c for c in continuous_candidates
        if c in df.columns and df[c].isnull().any()
    ]

    if continuous_cols:
        print(f"\n      KNN-imputing (k={KNN_K}): {continuous_cols}")
        scaler  = StandardScaler()
        imputer = KNNImputer(n_neighbors=KNN_K)
        vals          = df[continuous_cols].values.astype(float)
        vals_scaled   = scaler.fit_transform(vals)
        vals_imputed  = imputer.fit_transform(vals_scaled)
        vals_restored = scaler.inverse_transform(vals_imputed)
        df[continuous_cols] = vals_restored
        print("      KNN imputation complete. OK")
    else:
        print("      No continuous columns require KNN imputation. OK")

    remaining = df.isnull().sum().sum()
    print(f"\n      Nulls remaining: {remaining}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 — FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    race_progress_fraction = lap_number / total_laps_in_race

    Normalises lap number across races of different lengths.
    Monaco is ~78 laps, Spa is ~44. "Lap 40" means very different things
    strategically at those two circuits. "0.85 through the race" does not.
    """
    print("\n[7/8] Engineering features...")

    if "race_progress_fraction" not in df.columns:
        if "lap_number" in df.columns:
            total = df.groupby(
                ["season", "round_number"]
            )["lap_number"].transform("max")
            df["race_progress_fraction"] = df["lap_number"] / total
            print("      Added 'race_progress_fraction'.")
        else:
            print("      Cannot compute race_progress_fraction — lap_number missing.")
    else:
        print("      'race_progress_fraction' already present — skipping.")

    return df


# ─────────────────────────────────────────────────────────────────────────────
# STEP 8 — ONE-HOT ENCODING (only if raw string columns still exist)
# ─────────────────────────────────────────────────────────────────────────────

# We do NOT encode track_status here — it was decomposed into binary flags
# in Step 3. We do NOT encode driver/team/compound/race_name if the dataset
# was already output from a previous OHE run (columns like driver_VER exist).

CATEGORICAL_TO_ENCODE = [
    "current_compound",   # → current_compound_HARD / MEDIUM / SOFT
    "driver",             # → driver_VER, driver_HAM, ...
    "team",               # → team_Mercedes, team_Ferrari, ...
    "race_name",          # → race_name_British Grand Prix, ...
]


def one_hot_encode(df: pd.DataFrame) -> pd.DataFrame:
    """
    Only encode columns that still exist as raw strings.
    If the dataset was already processed (e.g. loaded from a previously
    saved CSV), these columns will already be binary and we skip encoding.
    """
    print("\n[8/8] One-hot encoding (raw categorical columns only)...")

    cols_to_encode = [c for c in CATEGORICAL_TO_ENCODE if c in df.columns]

    if not cols_to_encode:
        print("      No raw categorical columns found — already encoded or absent.")
        print("      Current binary compound columns present:")
        comp_cols = [c for c in df.columns if c.startswith("current_compound_")]
        for c in comp_cols:
            print(f"        {c}")
        return df

    print(f"      Encoding: {cols_to_encode}")
    before = len(df.columns)
    df = pd.get_dummies(df, columns=cols_to_encode, dtype=int)
    after = len(df.columns)
    print(f"      Columns: {before} -> {after}  (+{after - before} from OHE)")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# FINAL REPORT
# ─────────────────────────────────────────────────────────────────────────────

def final_report(df: pd.DataFrame):
    print("\n" + "=" * 60)
    print("  Final Dataset Summary")
    print("=" * 60)
    print(f"  Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

    # Target variable summary
    if "pit_this_lap" in df.columns:
        n0       = (df["pit_this_lap"] == 0).sum()
        n1       = (df["pit_this_lap"] == 1).sum()
        pit_rate = df["pit_this_lap"].mean() * 100
        print(f"\n  Task A — pit_this_lap")
        print(f"    Class 0 (no pit) : {n0:,}")
        print(f"    Class 1 (pit)    : {n1:,}")
        print(f"    Pit stop rate    : {pit_rate:.2f}%")
        if pit_rate < 10:
            print("    NOTE: Apply SMOTE at model training time (not here).")

    if "next_tire_compound" in df.columns:
        print(f"\n  Task B — next_tire_compound (pit laps only)")
        if "pit_this_lap" in df.columns:
            pit_rows = df[df["pit_this_lap"] == 1]
        else:
            pit_rows = df[df["next_tire_compound"].notna()]
        dist = pit_rows["next_tire_compound"].value_counts(dropna=False)
        for val, cnt in dist.items():
            print(f"    {str(val):<12}: {cnt:,}")

    # Column inventory
    meta_cols    = [c for c in ["season", "round_number"] if c in df.columns]
    target_cols  = [c for c in ["pit_this_lap", "next_tire_compound"] if c in df.columns]
    feature_cols = [c for c in df.columns if c not in meta_cols + target_cols]

    print(f"\n  Column inventory")
    print(f"    Metadata (exclude from model) : {meta_cols}")
    print(f"    Targets                       : {target_cols}")
    print(f"    Model features                : {len(feature_cols)}")

    # Group features for clarity
    continuous = [
        "lap_number", "lap_time", "lap_time_delta_prev", "tire_age_laps",
        "laps_since_last_pit", "avg_lap_time_on_stint", "avg_pit_time_team",
        "cars_within_2s_ahead", "cars_within_2s_behind", "position",
        "total_pit_stops_so_far", "stint_number", "race_progress_fraction",
    ]
    binary = [
        "is_new_tire", "is_being_attacked", "is_stuck_in_train",
        "sc_active", "vsc_active", "yellow_active", "red_flag_active",
    ]
    ohe = [c for c in feature_cols if c not in continuous + binary]

    print(f"\n    Continuous ({len([c for c in continuous if c in feature_cols])}):")
    for c in continuous:
        if c in df.columns:
            print(f"      {c}")

    print(f"\n    Binary flags ({len([c for c in binary if c in feature_cols])}):")
    for c in binary:
        if c in df.columns:
            print(f"      {c}")

    print(f"\n    One-hot encoded ({len(ohe)}) — first 10 shown:")
    for c in ohe[:10]:
        print(f"      {c}")
    if len(ohe) > 10:
        print(f"      ... and {len(ohe) - 10} more")

    print("\n  Preprocessing complete.")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def preprocess(
    input_file: str = INPUT_FILE,
    save_csv:   bool = SAVE_CSV,
) -> pd.DataFrame:

    print("=" * 60)
    print("  F1 Dataset Preprocessing Pipeline")
    print("=" * 60)

    df = load_dataset(input_file)
    df = drop_columns(df)
    df = decompose_track_status(df)
    df = filter_compounds(df)
    df = sort_and_validate(df)
    df = handle_missing_values(df)
    df = engineer_features(df)
    df = one_hot_encode(df)
    final_report(df)

    if save_csv:
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"\n  Saved to '{OUTPUT_FILE}'")

    return df


if __name__ == "__main__":
    df_preprocessed = preprocess(
        input_file=INPUT_FILE,
        save_csv=SAVE_CSV,
    )

  F1 Dataset Preprocessing Pipeline
  NOTE: Run this on the RAW CSV from dataset_builder.py.
        Do not chain-run on a previously preprocessed CSV.
[1/9] Loading dataset from 'f1_complete_dataset_2020_2024.csv'...
      Loaded 114,626 rows x 39 columns.

[2/9] Decomposing track_status into binary flags...
      sc_active=1        : 5,089 laps
      vsc_active=1       : 128 laps
      yellow_active=1    : 5,991 laps
      red_flag_active=1  : 1,900 laps
      All clear (0,0,0,0): 103,678 laps

[3/9] Dropping columns...
      DROP  'sector1_time'                       Requested drop; missingness and redundant with lap_time.
      DROP  'sector2_time'                       Requested drop; missingness and redundant with lap_time.
      DROP  'sector3_time'                       Requested drop; missingness and redundant with lap_time.
      DROP  'personal_best'                      Requested drop; binary, rare, very low signal-to-noise.
      DROP  'gap_to_car_ahead'                   

In [12]:
"""
F1 Pit Stop Prediction — Bi-LSTM  [v4 — Index Alignment Fixed]
================================================================
Replicates and improves on:
    Sasikumar, Leema & Balakrishnan (2025)
    "Data-driven pit stop decision support for Formula 1 using deep learning"
    Frontiers in Artificial Intelligence, 8:1673148

Root cause identified in v3
-----------------------------
The sequence builder used df.index.get_indexer(group_idx) to map
DataFrame indices to positions in the imputed numpy array X. But after
temporal_split(), train_df retains its original CSV row numbers as the
index (e.g. rows 0, 1, 5, 7, 12...) which are not the same as integer
positions in X_train_imp (which runs 0, 1, 2, 3, 4...). This mismatch
caused rows from different drivers and races to silently end up in the
same sequence window — exactly the boundary problem FIX 1 was meant to
solve, but re-introduced through the index bug.

Fix: reset_index(drop=True) on both splits immediately after temporal
split. This makes DataFrame positions match numpy array positions exactly.

All previous fixes retained
-----------------------------
FIX 1: Sequences per (race, driver) — now actually working correctly
FIX 2: No SMOTE — class weights only
FIX 3: avg_pit_time_team corrupted values → NaN
FIX 4: Redundant features dropped
FIX 5: Sparse OHE columns consolidated
FIX 6: lap_time_delta_prev edge cases → NaN

Training changes from v3
--------------------------
DROPOUT restored to paper values [0.2, 0.3, 0.3] — the low dropout in
v3 contributed to overfitting (train_loss 0.25, val_loss 0.39 by epoch 18).
The earlier reason for reducing dropout was the index bug causing
artificial val_loss < train_loss, not genuine over-regularisation.

RECURRENT_DROP restored to [0.2, 0.2, 0.2].

CLASS_WEIGHTS reduced to {0:1.0, 1:5.0} — v3's 1449 false positives
showed weight=8 was still too aggressive without SMOTE. Start at 5,
tune upward if recall is too low.
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score,
    precision_recall_curve, roc_curve,
    auc, balanced_accuracy_score,
)

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings("ignore")
os.makedirs("outputs", exist_ok=True)

# ═════════════════════════════════════════════════════════════════════════════
# HYPERPARAMETERS — change these to tune the model
# ═════════════════════════════════════════════════════════════════════════════

INPUT_FILE       = "f1_complete_preprocessed_dataset_2020_2024.csv"
TEST_YEAR        = 2024
TEST_LAST_N      = 8

SEQUENCE_LEN     = 10
# Try 15 or 20 if model underfits — longer context helps with slow degradation.

LSTM_UNITS       = [512, 256, 128]
# Reduce to [128, 64, 32] [256, 128, 64] if overfitting. Increase to [512, 256, 128] if underfitting.

DROPOUT_RATES    = [0.2, 0.3, 0.3]
# Paper values, restored from v3.
# Increase if val_loss >> train_loss (overfitting).
# Decrease if both losses are high (underfitting).

RECURRENT_DROP   = [0.2, 0.2, 0.2]
# Paper values, restored from v3. Same tuning logic as DROPOUT_RATES.

LEARNING_RATE    = 1e-4
# v3 value retained — 5e-4 caused early divergence.
# Try 2e-4 if convergence is too slow. Try 5e-5 if val_loss still diverges.

BATCH_SIZE       = 64
# Try 128 for smoother gradients if training is noisy.
# Try 32 if batch=64 misses too many pit stop sequences per batch.

CLASS_WEIGHTS    = {0: 1.0, 1: 25.0}
# Reduced from v3's 8.0 which caused 1449 false positives (precision=0.069).
# TUNING GUIDE:
#   precision < 0.3 and FP >> TP  → reduce toward 3.0
#   recall    < 0.4 and FN >> TP  → increase toward 8.0
#   both above 0.5                → sweet spot, try 0.5 steps either way

MAX_EPOCHS       = 50
ES_PATIENCE      = 12
# How many epochs of no val_loss improvement before stopping.
# With LR=1e-4, model moves slowly — 12 gives it time to benefit from LR drops.

LR_PATIENCE      = 6
# Epochs before halving the LR. With ES_PATIENCE=12 this allows two
# LR reductions (6+6=12 epochs total) before early stopping fires.

LR_FACTOR        = 0.5
LR_MIN           = 1e-7
VAL_SPLIT        = 0.20

MIN_DRIVER_LAPS  = 500
TEAM_RENAME_MAP  = {
    "team_Alfa Romeo Racing" : "team_Kick Sauber",
    "team_Alfa Romeo"        : "team_Kick Sauber",
    "team_AlphaTauri"        : "team_RB",
    "team_Racing Point"      : "team_Aston Martin",
    "team_Renault"           : "team_Alpine",
}

METADATA_COLS    = ["season", "round_number"]
TARGET_A         = "pit_this_lap"
TARGET_B         = "next_tire_compound"
FEATURES_TO_DROP = [
    "avg_pit_time_team",    # corrupted: builder bug produces negative values
    "stint_number",         # redundant: same info as total_pit_stops_so_far
    "is_new_tire",          # redundant: derivable from tire_age_laps == 1
    "laps_since_last_pit",  # redundant: same info as tire_age_laps
    "lap_number",           # redundant: race_progress_fraction is better
]

N_BOOTSTRAP      = 1000
RANDOM_SEED      = 42

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — LOAD & VALIDATE
# ─────────────────────────────────────────────────────────────────────────────

def load_and_validate(path: str) -> pd.DataFrame:
    """
    Load the preprocessed CSV and apply FIX 3 immediately:
    avg_pit_time_team values outside [1.5, 60] seconds are physically
    impossible pit durations caused by a computation bug in the dataset
    builder. Set them to NaN so KNN imputer can replace them.
    """
    print(f"\n{'='*62}")
    print(f"[1/8] Loading: {path}")
    df = pd.read_csv(path, low_memory=False)
    print(f"      Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

    if "avg_pit_time_team" in df.columns:
        bad = (df["avg_pit_time_team"] < 1.5) | (df["avg_pit_time_team"] > 60)
        if bad.sum() > 0:
            print(f"      FIX 3: {bad.sum():,} impossible avg_pit_time_team → NaN")
            df.loc[bad, "avg_pit_time_team"] = np.nan

    for col in [TARGET_A, "season", "round_number"]:
        assert col in df.columns, f"Required column '{col}' missing."

    n0, n1 = (df[TARGET_A] == 0).sum(), (df[TARGET_A] == 1).sum()
    print(f"      pit_this_lap=0: {n0:,}  |  pit_this_lap=1: {n1:,}")
    print(f"      Pit rate: {df[TARGET_A].mean()*100:.2f}%")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — CONSOLIDATE SPARSE OHE COLUMNS  (FIX 4 + 5)
# ─────────────────────────────────────────────────────────────────────────────

def consolidate_sparse_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    FIX 4: Merge renamed team columns into the current team name.
    The same physical team appears under different names across seasons
    (e.g. AlphaTauri 2020-2023, RB 2024). Two sparse columns for one
    team dilutes the signal — merging them gives a denser, accurate column.

    FIX 5: Collapse infrequent driver columns into driver_OTHER.
    Drivers below MIN_DRIVER_LAPS laps have columns that are 99%+ zeros.
    Merging them preserves the "substitute/rare driver" signal without
    adding dozens of near-empty dimensions to the feature space.
    """
    print("\n[2/8] Consolidating sparse OHE columns...")

    for old, new in TEAM_RENAME_MAP.items():
        if old in df.columns:
            if new in df.columns:
                df[new] = df[[new, old]].max(axis=1)
            else:
                df[new] = df[old]
            df.drop(columns=[old], inplace=True)
            print(f"      Team merge: '{old}' → '{new}'")

    driver_cols       = [c for c in df.columns if c.startswith("driver_")]
    driver_lap_counts = {c: int(df[c].sum()) for c in driver_cols}
    sparse            = [c for c, n in driver_lap_counts.items() if n < MIN_DRIVER_LAPS]

    if sparse:
        print(f"      Collapsing {len(sparse)} sparse drivers → driver_OTHER:")
        for c in sparse:
            print(f"        {c}: {driver_lap_counts[c]:,} laps")
        df["driver_OTHER"] = df[sparse].max(axis=1)
        df.drop(columns=sparse, inplace=True)

    d = len([c for c in df.columns if c.startswith("driver_")])
    t = len([c for c in df.columns if c.startswith("team_")])
    print(f"      Driver cols: {d}  |  Team cols: {t}  |  Total: {len(df.columns)}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — TEMPORAL SPLIT + INDEX RESET  (ROOT CAUSE FIX)
# ─────────────────────────────────────────────────────────────────────────────

def temporal_split(df: pd.DataFrame) -> tuple:
    """
    Split into train (all races except final 8 of 2024) and test
    (final 8 races of 2024), matching the paper exactly.

    CRITICAL — reset_index(drop=True) on both splits.
    ───────────────────────────────────────────────────
    After slicing the DataFrame, the retained rows keep their original
    CSV row numbers as the index. For example train_df might have index
    values [0, 1, 2, 5, 7, 12, ...] — non-contiguous integers.

    impute_and_scale() converts train_df to a numpy array X_train_imp.
    Numpy arrays are always 0-indexed: row 0 of X_train_imp corresponds
    to the FIRST row of train_df, row 1 to the SECOND row, regardless
    of what the DataFrame index says.

    In build_sequences_per_group(), we call:
        pos = df.index.get_indexer(group_idx)
    get_indexer() returns the POSITION of each label in df.index.
    If df.index = [0,1,2,5,7,12,...] and group_idx = [5,7,12], then
    get_indexer returns [3, 4, 5] — the correct positions in df.
    BUT if df.index was NOT reset and X was built from the original
    sliced df, these positions map correctly.

    The problem occurred in v3 because train_df was passed to
    impute_and_scale() before reset_index(), then the RESET happened
    inside build_sequences_per_group() indirectly — causing a mismatch
    between the positions used to index X and the actual row order.

    Solution: reset both splits HERE, once, before anything else runs.
    After reset: df.index = [0, 1, 2, 3, ...] matches X[0], X[1], X[2]...
    """
    print(f"\n[3/8] Temporal split (test = last {TEST_LAST_N} races of {TEST_YEAR})...")

    test_rounds = sorted(
        df[df["season"] == TEST_YEAR]["round_number"].unique()
    )[-TEST_LAST_N:]

    mask     = (df["season"] == TEST_YEAR) & (df["round_number"].isin(test_rounds))
    train_df = df[~mask].copy().reset_index(drop=True)   # ← ROOT CAUSE FIX
    test_df  = df[mask].copy().reset_index(drop=True)    # ← ROOT CAUSE FIX

    print(f"      Test rounds  : {test_rounds}")
    print(f"      Train laps   : {len(train_df):,}  (index 0..{len(train_df)-1})")
    print(f"      Test laps    : {len(test_df):,}   (index 0..{len(test_df)-1})")
    print(f"      Train pit %  : {train_df[TARGET_A].mean()*100:.2f}%")
    print(f"      Test pit %   : {test_df[TARGET_A].mean()*100:.2f}%")
    return train_df, test_df


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — FEATURE COLUMNS & EDGE CASE FIXES
# ─────────────────────────────────────────────────────────────────────────────

def get_feature_columns(df: pd.DataFrame) -> list:
    """
    Return the model input feature list by excluding metadata, targets,
    and explicitly noisy/redundant columns (FEATURES_TO_DROP).
    Missing columns in FEATURES_TO_DROP are silently skipped.
    """
    exclude = set(METADATA_COLS + [TARGET_A, TARGET_B] + FEATURES_TO_DROP)
    return [c for c in df.columns if c not in exclude]


def fix_delta_laptime_edges(train_df: pd.DataFrame,
                             test_df:  pd.DataFrame) -> tuple:
    """
    FIX 6 — Set structurally misleading lap_time_delta_prev to NaN.

    Case 1 — Lap 1 of each race:
        No previous lap exists. Any value here is meaningless context.
        Identified by lap_number == 1.

    Case 2 — Lap immediately after a pit stop:
        The pit lap is ~20s slower than a normal lap. The very next lap
        on fresh tyres looks like a massive improvement (e.g. -22 seconds
        delta) which is purely a pit stop artefact, not real performance.
        Identified by shifting pit_this_lap forward by 1 row.

    NOTE: shift(1) crosses driver boundaries in the sorted DataFrame but
    this is harmless — the first lap of a new driver is already flagged
    by Case 1 (lap_number == 1) and gets NaN from that rule.

    KNN imputer replaces all NaN with context-appropriate interpolations.
    """
    if "lap_time_delta_prev" not in train_df.columns:
        print("\n[4/8] lap_time_delta_prev not found — skipping FIX 6.")
        return train_df, test_df

    print("\n[4/8] Fixing lap_time_delta_prev edge cases (FIX 6)...")
    total_1, total_pp = 0, 0

    for df in [train_df, test_df]:
        if "lap_number" in df.columns:
            m = df["lap_number"] == 1
            df.loc[m, "lap_time_delta_prev"] = np.nan
            total_1 += m.sum()

        pp = df[TARGET_A].shift(1, fill_value=0) == 1
        df.loc[pp, "lap_time_delta_prev"] = np.nan
        total_pp += pp.sum()

    print(f"      Lap-1 NaN    : {total_1:,}")
    print(f"      Post-pit NaN : {total_pp:,}")
    return train_df, test_df


# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — IMPUTATION & SCALING
# ─────────────────────────────────────────────────────────────────────────────

def impute_and_scale(train_df:     pd.DataFrame,
                     test_df:      pd.DataFrame,
                     feature_cols: list) -> tuple:
    """
    Paper's two-stage imputation — fitted on training data only.

    Stage 1 — StandardScaler (fit on train):
        Normalises to mean=0, std=1. Required before KNN because Euclidean
        distance is scale-sensitive: without scaling, lap_time (~80s)
        completely dominates position (1-20) in neighbour calculations.

    Stage 2 — KNNImputer k=5 (fit on train):
        Replaces NaN with the mean of the 5 most similar laps in
        normalised feature space. Preserves multivariate relationships
        far better than mean/median imputation.

    Fitting on train only prevents test set statistics (means, variances,
    neighbour structure) from leaking into the training pipeline.

    The output arrays are z-score normalised — this is what feeds
    directly into the Bi-LSTM. No inverse transform is applied.
    """
    print(f"\n[5/8] Imputing and scaling (fit on train only)...")
    X_tr = train_df[feature_cols].values.astype(float)
    X_te = test_df[feature_cols].values.astype(float)

    print(f"      NaN in train: {np.isnan(X_tr).sum():,}  "
          f"|  NaN in test: {np.isnan(X_te).sum():,}")

    scaler  = StandardScaler()
    imputer = KNNImputer(n_neighbors=5)

    X_tr_s = scaler.fit_transform(X_tr)
    X_tr_i = imputer.fit_transform(X_tr_s)
    X_te_s = scaler.transform(X_te)
    X_te_i = imputer.transform(X_te_s)

    print(f"      Done. Shape: train={X_tr_i.shape}, test={X_te_i.shape}")
    return X_tr_i, X_te_i, scaler, imputer


# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 — SEQUENCE CREATION PER (RACE, DRIVER)  [FIX 1 — now correct]
# ─────────────────────────────────────────────────────────────────────────────

def _sequences_from_array(X: np.ndarray,
                           y: np.ndarray,
                           seq_len: int) -> tuple:
    """
    Build overlapping windows of length seq_len from a contiguous array
    belonging to a single (race, driver) group.

    Sequence i covers laps [i : i+seq_len] and its label is y[i+seq_len-1]
    — did this driver pit at the end of the last lap in this window?
    A group with N laps produces max(0, N - seq_len) sequences.
    """
    if len(X) <= seq_len:
        return np.empty((0, seq_len, X.shape[1]), dtype=np.float32), \
               np.empty(0, dtype=np.int32)
    Xs = np.array([X[i: i + seq_len] for i in range(len(X) - seq_len)])
    ys = np.array([y[i + seq_len - 1] for i in range(len(X) - seq_len)])
    return Xs, ys


def build_sequences_per_group(train_df:      pd.DataFrame,
                               test_df:      pd.DataFrame,
                               X_train:      np.ndarray,
                               X_test:       np.ndarray,
                               feature_cols: list) -> tuple:
    """
    FIX 1 — Build sequences strictly within each (season, round, driver).

    WHY this works now (v3 had the same intention but a bug):
        After reset_index(drop=True) in temporal_split(), train_df.index
        = [0, 1, 2, ..., N-1]. The numpy array X_train has the same
        ordering: X_train[i] = features of train_df.iloc[i].

        When we call df.groupby(...).groups, the returned group_idx
        values are DataFrame index labels. After reset_index, these
        labels ARE the integer positions in X — so X[group_idx] is
        always correct with no get_indexer() needed.

    How sequences are built:
        1. Group train_df by (season, round_number) — one race at a time.
        2. Within each race, identify drivers from OHE driver columns.
        3. For each driver in each race, extract their rows (in lap order)
           from X and y, build windows, append to the output lists.
        4. Concatenate all groups at the end.

    Drivers with fewer than SEQUENCE_LEN laps produce zero sequences.
    This is correct — a driver who retired on lap 7 with SEQUENCE_LEN=10
    never had 10 consecutive laps and cannot contribute valid sequences.
    """
    print(f"\n[6/8] Building sequences per (race, driver) [FIX 1 — corrected]...")

    driver_cols = [c for c in feature_cols if c.startswith("driver_")]

    def _build(df: pd.DataFrame, X: np.ndarray) -> tuple:
        all_X, all_y = [], []
        n_groups, n_seq_total = 0, 0

        for (season, rnd), grp_idx in df.groupby(["season", "round_number"]).groups.items():
            # grp_idx are DataFrame index labels = integer positions after reset_index
            pos    = grp_idx.values           # numpy array of integer positions
            X_race = X[pos]                   # rows for this race, all drivers
            y_race = df.loc[grp_idx, TARGET_A].values.astype(int)

            if driver_cols:
                # Identify which driver each row belongs to via OHE columns
                drv_labels = df.loc[grp_idx, driver_cols].idxmax(axis=1).values

                for drv in np.unique(drv_labels):
                    mask  = (drv_labels == drv)
                    Xs, ys = _sequences_from_array(X_race[mask], y_race[mask], SEQUENCE_LEN)
                    if len(Xs) > 0:
                        all_X.append(Xs)
                        all_y.append(ys)
                        n_seq_total += len(Xs)
            else:
                # Fallback: no driver columns — per-race sequences only
                Xs, ys = _sequences_from_array(X_race, y_race, SEQUENCE_LEN)
                if len(Xs) > 0:
                    all_X.append(Xs)
                    all_y.append(ys)
                    n_seq_total += len(Xs)

            n_groups += 1

        X_out = np.concatenate(all_X) if all_X else np.empty((0, SEQUENCE_LEN, X.shape[1]))
        y_out = np.concatenate(all_y) if all_y else np.empty(0, dtype=int)
        return X_out, y_out

    X_tr_seq, y_tr_seq = _build(train_df, X_train)
    X_te_seq, y_te_seq = _build(test_df,  X_test)

    print(f"      Train sequences : {X_tr_seq.shape}  "
          f"pit rate: {y_tr_seq.mean()*100:.2f}%")
    print(f"      Test  sequences : {X_te_seq.shape}  "
          f"pit rate: {y_te_seq.mean()*100:.2f}%")

    return X_tr_seq, y_tr_seq, X_te_seq, y_te_seq


# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 — BUILD BI-LSTM  (paper architecture, paper dropout)
# ─────────────────────────────────────────────────────────────────────────────

def build_bilstm(input_shape: tuple) -> Model:
    """
    Paper's exact 3-layer Bi-LSTM (Table 1, Figure 6).
    Dropout restored to paper values [0.2, 0.3, 0.3] — reduced in v3
    to diagnose a suspected under-regularisation issue, but the real
    cause was the index bug. With correct sequences the model can
    overfit real temporal patterns, so full regularisation is needed.

    Architecture:
        BiLSTM 256 → Dropout 0.2 → BiLSTM 128 → Dropout 0.3
        → BiLSTM 64 → Dropout 0.3 → Dense 1 sigmoid

    Why Bidirectional:
        Forward pass sees laps 1→N (degradation building up).
        Backward pass sees laps N→1 (what happens AFTER each lap).
        Together the model learns that a lap "looks different" when it
        comes just before a pit stop — something a unidirectional LSTM
        cannot see until it's too late.
    """
    inp = Input(shape=input_shape, name="lap_sequence_input")

    x = Bidirectional(
        LSTM(LSTM_UNITS[0], return_sequences=True,
             recurrent_dropout=RECURRENT_DROP[0]),
        name="BiLSTM_256")(inp)
    x = Dropout(DROPOUT_RATES[0], name="Drop_1")(x)

    x = Bidirectional(
        LSTM(LSTM_UNITS[1], return_sequences=True,
             recurrent_dropout=RECURRENT_DROP[1]),
        name="BiLSTM_128")(x)
    x = Dropout(DROPOUT_RATES[1], name="Drop_2")(x)

    x = Bidirectional(
        LSTM(LSTM_UNITS[2], return_sequences=False,
             recurrent_dropout=RECURRENT_DROP[2]),
        name="BiLSTM_64")(x)
    x = Dropout(DROPOUT_RATES[2], name="Drop_3")(x)

    out = Dense(1, activation="sigmoid", name="pit_probability")(x)

    model = Model(inputs=inp, outputs=out, name="Bi_LSTM_F1_v4")
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


# ─────────────────────────────────────────────────────────────────────────────
# STEP 8 — TRAIN
# ─────────────────────────────────────────────────────────────────────────────

def train_model(X_tr_seq: np.ndarray,
                y_tr_seq: np.ndarray) -> tuple:
    """
    Train with paper callbacks and tuned class weights.

    EarlyStopping (patience=12):
        Monitors val_loss. 12 epochs patience gives the model time to
        benefit from two LR reductions before stopping.

    ReduceLROnPlateau (patience=6, factor=0.5):
        Halves LR after 6 epochs of plateau. The sequence is:
          → 6 epochs no improvement → LR halved
          → 6 more epochs → LR halved again
          → 12 total no-improvement epochs → early stop
        This gives the model two "rescue" opportunities via LR reduction.

    class_weight = {0:1.0, 1:5.0}:
        Replaces SMOTE. Every missed pit stop costs 5x a missed non-pit.
        Lower than v3's 8.0 to reduce the 1449 false positives seen there.
        The correctly-built sequences should allow the model to learn
        real pit stop patterns rather than firing on every ambiguous lap.
    """
    print(f"\n[7/8] Training Bi-LSTM...")
    print(f"      Train sequences: {X_tr_seq.shape}  "
          f"pit rate: {y_tr_seq.mean()*100:.2f}%")
    print(f"      LR={LEARNING_RATE}  batch={BATCH_SIZE}  "
          f"class_weights={CLASS_WEIGHTS}  max_epochs={MAX_EPOCHS}")

    model = build_bilstm((SEQUENCE_LEN, X_tr_seq.shape[2]))
    model.summary()

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=ES_PATIENCE,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=LR_FACTOR,
                          patience=LR_PATIENCE, min_lr=LR_MIN, verbose=1),
    ]

    history = model.fit(
        X_tr_seq, y_tr_seq,
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VAL_SPLIT,
        class_weight=CLASS_WEIGHTS,
        callbacks=callbacks,
        verbose=1,
    )

    actual = len(history.history["loss"])
    best   = int(np.argmin(history.history["val_loss"])) + 1
    best_v = min(history.history["val_loss"])
    print(f"\n      Epochs run: {actual}  |  Best: epoch {best}  "
          f"(val_loss={best_v:.4f})")
    return model, history


# ─────────────────────────────────────────────────────────────────────────────
# EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

def find_optimal_threshold(model:       Model,
                            X_tr_seq:   np.ndarray,
                            y_tr_seq:   np.ndarray) -> float:
    """
    Find the threshold that maximises F1 on training predictions.
    Applied to test predictions — no test data used here (no leakage).

    Paper: 'an optimal classification threshold was computed from the
    precision recall curve which enabled better F1-score alignment.'
    """
    probs          = model.predict(X_tr_seq, verbose=0).flatten()
    p, r, thr      = precision_recall_curve(y_tr_seq, probs)
    f1_arr         = np.where((p + r) == 0, 0, 2*p*r/(p+r))
    best           = int(np.argmax(f1_arr[:-1]))
    t              = float(thr[best])
    print(f"\n      Optimal threshold: {t:.4f}  "
          f"(train: precision={p[best]:.3f}, recall={r[best]:.3f}, "
          f"F1={f1_arr[best]:.3f})")
    return t


def bootstrap_ci(y_true: np.ndarray,
                 y_pred: np.ndarray,
                 y_prob: np.ndarray) -> dict:
    """
    95% bootstrap CIs for F1, Balanced Accuracy, ROC-AUC, AUC-PR.
    1,000 resamples with replacement (paper's method).
    """
    rng = np.random.RandomState(RANDOM_SEED)
    f1s, bas, rocs, prs = [], [], [], []
    for _ in range(N_BOOTSTRAP):
        idx = rng.randint(0, len(y_true), len(y_true))
        yt, yb, yp = y_true[idx], y_pred[idx], y_prob[idx]
        if yt.sum() == 0 or yt.sum() == len(yt):
            continue
        f1s.append(f1_score(yt, yb, zero_division=0))
        bas.append(balanced_accuracy_score(yt, yb))
        rocs.append(roc_auc_score(yt, yp))
        pp, rr, _ = precision_recall_curve(yt, yp)
        prs.append(auc(rr, pp))

    def _ci(a):
        a = np.array(a)
        return float(a.mean()), float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))

    return {"F1": _ci(f1s), "Balanced Accuracy": _ci(bas),
            "ROC-AUC": _ci(rocs), "AUC-PR": _ci(prs)}


def evaluate(model:     Model,
             X_te_seq:  np.ndarray,
             y_te_seq:  np.ndarray,
             X_tr_seq:  np.ndarray,
             y_tr_seq:  np.ndarray,
             history) -> dict:
    """
    Full paper metric suite: Precision, Recall, F1, Specificity,
    Balanced Accuracy, ROC-AUC, AUC-PR with 95% bootstrap CIs.
    Threshold determined from training data PR curve only.
    """
    print(f"\n[8/8] Evaluating on test set...")

    y_prob    = model.predict(X_te_seq, verbose=0).flatten()
    threshold = find_optimal_threshold(model, X_tr_seq, y_tr_seq)
    y_pred    = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_te_seq, y_pred).ravel()
    precision      = precision_score(y_te_seq, y_pred, zero_division=0)
    recall         = recall_score(y_te_seq, y_pred, zero_division=0)
    f1             = f1_score(y_te_seq, y_pred, zero_division=0)
    specificity    = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    bal_acc        = balanced_accuracy_score(y_te_seq, y_pred)
    roc_auc        = roc_auc_score(y_te_seq, y_prob)
    p_arr, r_arr, _= precision_recall_curve(y_te_seq, y_prob)
    auc_pr         = auc(r_arr, p_arr)

    print("      Computing 95% bootstrap CIs...")
    ci = bootstrap_ci(y_te_seq, y_pred, y_prob)

    print("\n" + "="*62)
    print("  EVALUATION REPORT — Bi-LSTM  (minority class: pit=1)")
    print("="*62)
    print(f"  Threshold                : {threshold:.4f}")
    print()
    print(f"  {'Metric':<24} {'Score':>7}   {'95% CI':>22}")
    print(f"  {'-'*58}")
    print(f"  {'Precision':<24} {precision:>7.3f}")
    print(f"  {'Recall':<24} {recall:>7.3f}")
    print(f"  {'F1-Score':<24} {f1:>7.3f}   [{ci['F1'][1]:.3f}, {ci['F1'][2]:.3f}]")
    print(f"  {'Specificity':<24} {specificity:>7.3f}")
    print(f"  {'Balanced Accuracy':<24} {bal_acc:>7.3f}   "
          f"[{ci['Balanced Accuracy'][1]:.3f}, {ci['Balanced Accuracy'][2]:.3f}]")
    print(f"  {'ROC-AUC':<24} {roc_auc:>7.3f}   "
          f"[{ci['ROC-AUC'][1]:.3f}, {ci['ROC-AUC'][2]:.3f}]")
    print(f"  {'AUC-PR':<24} {auc_pr:>7.3f}   "
          f"[{ci['AUC-PR'][1]:.3f}, {ci['AUC-PR'][2]:.3f}]")
    print()
    print(f"  Confusion Matrix (threshold={threshold:.4f}):")
    print(f"  {'':24} Pred 0    Pred 1")
    print(f"  {'Actual 0 (no pit)':<24} {tn:>6}    {fp:>6}")
    print(f"  {'Actual 1 (pit)':<24} {fn:>6}    {tp:>6}")
    print("="*62)

    return dict(
        threshold=threshold, precision=precision, recall=recall,
        f1=f1, specificity=specificity, balanced_accuracy=bal_acc,
        roc_auc=roc_auc, auc_pr=auc_pr,
        tn=tn, fp=fp, fn=fn, tp=tp, ci=ci,
        y_prob=y_prob, y_pred=y_pred, y_true=y_te_seq,
        p_arr=p_arr, r_arr=r_arr,
    )


# ─────────────────────────────────────────────────────────────────────────────
# PLOTS
# ─────────────────────────────────────────────────────────────────────────────

def plot_all(metrics: dict, history) -> None:
    """6 evaluation plots matching the paper's figure suite."""
    print("\n      Generating plots...")
    CB = "#1f77b4"; CO = "#ff7f0e"; CG = "#2ca02c"; CP = "#9467bd"; CR = "#8c564b"

    # 1 — Training curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    best_ep = int(np.argmin(history.history["val_loss"]))
    for ax, (tk, vk), title in zip(
        axes,
        [("loss", "val_loss"), ("accuracy", "val_accuracy")],
        ["Loss (Binary Cross-Entropy)", "Accuracy"]
    ):
        ax.plot(history.history[tk], label="Train", color=CB)
        ax.plot(history.history[vk], label="Val",   color=CO)
        ax.axvline(best_ep, color="gray", linestyle=":", alpha=0.7,
                   label=f"Best epoch ({best_ep+1})")
        ax.set_title(title); ax.set_xlabel("Epoch")
        ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle("Bi-LSTM Training Curves  [v4]", y=1.01)
    plt.tight_layout()
    plt.savefig("outputs/01_training_curves.png", dpi=150, bbox_inches="tight")
    plt.close()

    # 2 — Confusion matrix
    cm    = np.array([[metrics["tn"], metrics["fp"]],
                      [metrics["fn"], metrics["tp"]]])
    annot = np.array([["TN\n{:,}".format(metrics["tn"]),
                        "FP\n{:,}".format(metrics["fp"])],
                       ["FN\n{:,}".format(metrics["fn"]),
                        "TP\n{:,}".format(metrics["tp"])]])
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", ax=ax,
                xticklabels=["Pred: No Pit", "Pred: Pit"],
                yticklabels=["Actual: No Pit", "Actual: Pit"],
                linewidths=0.5, annot_kws={"size": 13})
    ax.set_title(f"Confusion Matrix  (threshold={metrics['threshold']:.4f})")
    plt.tight_layout()
    plt.savefig("outputs/02_confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.close()

    # 3 — PR curve
    noskill = metrics["y_true"].sum() / len(metrics["y_true"])
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(metrics["r_arr"], metrics["p_arr"], color=CB, lw=2,
            label=f"Bi-LSTM  AUC-PR={metrics['auc_pr']:.3f}")
    ax.axhline(noskill, color="gray", linestyle="--",
               label=f"No-Skill (prevalence={noskill:.3f})")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curve — Bi-LSTM  [v4]")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("outputs/03_pr_curve.png", dpi=150, bbox_inches="tight")
    plt.close()

    # 4 — ROC
    fpr, tpr, _ = roc_curve(metrics["y_true"], metrics["y_prob"])
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, color=CB, lw=2,
            label=f"Bi-LSTM  AUC={metrics['roc_auc']:.3f}")
    ax.plot([0,1],[0,1], color="gray", linestyle="--", label="Random Guess")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve — Bi-LSTM  [v4]")
    ax.legend(loc="lower right"); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("outputs/04_roc_curve.png", dpi=150, bbox_inches="tight")
    plt.close()

    # 5 — Metric bar chart
    bd = {"Precision": metrics["precision"], "Recall": metrics["recall"],
          "F1-Score": metrics["f1"], "Specificity": metrics["specificity"],
          "Balanced\nAcc.": metrics["balanced_accuracy"]}
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(bd.keys(), bd.values(),
                  color=[CB, CG, CO, CP, CR], width=0.5, edgecolor="white")
    for bar, val in zip(bars, bd.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.015,
                f"{val:.3f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_ylim(0, 1.15); ax.set_ylabel("Score"); ax.grid(axis="y", alpha=0.3)
    ax.set_title("Bi-LSTM — Minority Class Performance (pit_this_lap=1)  [v4]")
    plt.tight_layout()
    plt.savefig("outputs/05_metric_bar_chart.png", dpi=150, bbox_inches="tight")
    plt.close()

    # 6 — CI table
    ci = metrics["ci"]
    rows = [
        ["F1-Score",          f"{ci['F1'][0]:.3f}",
                              f"[{ci['F1'][1]:.3f}, {ci['F1'][2]:.3f}]"],
        ["Balanced Accuracy", f"{ci['Balanced Accuracy'][0]:.3f}",
                              f"[{ci['Balanced Accuracy'][1]:.3f}, {ci['Balanced Accuracy'][2]:.3f}]"],
        ["ROC-AUC",           f"{ci['ROC-AUC'][0]:.3f}",
                              f"[{ci['ROC-AUC'][1]:.3f}, {ci['ROC-AUC'][2]:.3f}]"],
        ["AUC-PR",            f"{ci['AUC-PR'][0]:.3f}",
                              f"[{ci['AUC-PR'][1]:.3f}, {ci['AUC-PR'][2]:.3f}]"],
    ]
    fig, ax = plt.subplots(figsize=(8, 2.5))
    ax.axis("off")
    tbl = ax.table(cellText=rows,
                   colLabels=["Metric", "Score",
                               "95% CI (1000 bootstrap resamples)"],
                   cellLoc="center", loc="center",
                   colColours=["#d0e4f7"]*3)
    tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 1.8)
    ax.set_title("Bi-LSTM — 95% Bootstrap CIs  [v4]", fontsize=11, pad=20)
    plt.tight_layout()
    plt.savefig("outputs/06_ci_table.png", dpi=150, bbox_inches="tight")
    plt.close()

    print("      6 plots saved to outputs/")


# ─────────────────────────────────────────────────────────────────────────────
# SAVE RESULTS
# ─────────────────────────────────────────────────────────────────────────────

def save_results(metrics: dict) -> None:
    """Write plain-text results in paper reporting format."""
    ci = metrics["ci"]
    lines = [
        "Bi-LSTM F1 Pit Stop Prediction — Results  [v4]",
        "=" * 54,
        f"Threshold: {metrics['threshold']:.4f}",
        "",
        "Core Metrics (minority class — pit_this_lap=1):",
        f"  Precision         : {metrics['precision']:.4f}",
        f"  Recall            : {metrics['recall']:.4f}",
        f"  F1-Score          : {metrics['f1']:.4f}  "
          f"CI [{ci['F1'][1]:.3f}, {ci['F1'][2]:.3f}]",
        f"  Specificity       : {metrics['specificity']:.4f}",
        f"  Balanced Accuracy : {metrics['balanced_accuracy']:.4f}  "
          f"CI [{ci['Balanced Accuracy'][1]:.3f}, {ci['Balanced Accuracy'][2]:.3f}]",
        f"  ROC-AUC           : {metrics['roc_auc']:.4f}  "
          f"CI [{ci['ROC-AUC'][1]:.3f}, {ci['ROC-AUC'][2]:.3f}]",
        f"  AUC-PR            : {metrics['auc_pr']:.4f}  "
          f"CI [{ci['AUC-PR'][1]:.3f}, {ci['AUC-PR'][2]:.3f}]",
        "",
        "Confusion Matrix:",
        f"  TN:{metrics['tn']:>5}  FP:{metrics['fp']:>5}",
        f"  FN:{metrics['fn']:>5}  TP:{metrics['tp']:>5}",
        "",
        "Hyperparameters:",
        f"  LEARNING_RATE  : {LEARNING_RATE}",
        f"  BATCH_SIZE     : {BATCH_SIZE}",
        f"  CLASS_WEIGHTS  : {CLASS_WEIGHTS}",
        f"  SEQUENCE_LEN   : {SEQUENCE_LEN}",
        f"  LSTM_UNITS     : {LSTM_UNITS}",
        f"  DROPOUT_RATES  : {DROPOUT_RATES}",
        f"  RECURRENT_DROP : {RECURRENT_DROP}",
        f"  ES_PATIENCE    : {ES_PATIENCE}",
        f"  LR_PATIENCE    : {LR_PATIENCE}",
    ]
    with open("outputs/results_summary.txt", "w") as f:
        f.write("\n".join(lines))
    print("      Results saved to outputs/results_summary.txt")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("=" * 62)
    print("  F1 Pit Stop Prediction — Bi-LSTM  [v4: Index Fix]")
    print("=" * 62)

    df                            = load_and_validate(INPUT_FILE)
    df                            = consolidate_sparse_columns(df)
    train_df, test_df             = temporal_split(df)          # reset_index here
    feature_cols                  = get_feature_columns(df)

    print(f"\n      Model features: {len(feature_cols)}")

    train_df, test_df             = fix_delta_laptime_edges(train_df, test_df)

    y_train                       = train_df[TARGET_A].values.astype(int)
    y_test                        = test_df[TARGET_A].values.astype(int)

    X_tr_imp, X_te_imp, _, _      = impute_and_scale(train_df, test_df, feature_cols)

    X_tr_seq, y_tr_seq, \
    X_te_seq, y_te_seq            = build_sequences_per_group(
                                        train_df, test_df,
                                        X_tr_imp, X_te_imp,
                                        feature_cols)

    model, history                = train_model(X_tr_seq, y_tr_seq)
    model.save("outputs/bilstm_f1_pitstop_v4.keras")
    print("      Model saved: outputs/bilstm_f1_pitstop_v4.keras")

    metrics                       = evaluate(model, X_te_seq, y_te_seq,
                                             X_tr_seq, y_tr_seq, history)
    plot_all(metrics, history)
    save_results(metrics)

    print("\n" + "=" * 62)
    print("  DONE. All outputs in outputs/")
    print("=" * 62)
    return model, metrics


if __name__ == "__main__":
    model, metrics = main()

  F1 Pit Stop Prediction — Bi-LSTM  [v4: Index Fix]

[1/8] Loading: f1_complete_preprocessed_dataset_2020_2024.csv
      Shape: 106,793 rows x 110 columns
      FIX 3: 106,764 impossible avg_pit_time_team → NaN
      pit_this_lap=0: 103,406  |  pit_this_lap=1: 3,387
      Pit rate: 3.17%

[2/8] Consolidating sparse OHE columns...
      Team merge: 'team_Alfa Romeo Racing' → 'team_Kick Sauber'
      Team merge: 'team_Alfa Romeo' → 'team_Kick Sauber'
      Team merge: 'team_AlphaTauri' → 'team_RB'
      Team merge: 'team_Racing Point' → 'team_Aston Martin'
      Team merge: 'team_Renault' → 'team_Alpine'
      Collapsing 6 sparse drivers → driver_OTHER:
        driver_AIT: 87 laps
        driver_BEA: 99 laps
        driver_COL: 365 laps
        driver_DOO: 57 laps
        driver_FIT: 139 laps
        driver_KUB: 121 laps
      Driver cols: 31  |  Team cols: 10  |  Total: 100

[3/8] Temporal split (test = last 8 races of 2024)...
      Test rounds  : [np.int64(16), np.int64(17), np.int64(

Model: "Bi_LSTM_F1_v4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lap_sequence_input (InputLayer) │ (None, 10, 91)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ BiLSTM_256 (Bidirectional)      │ (None, 10, 1024)       │     2,473,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Drop_1 (Dropout)                │ (None, 10, 1024)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ BiLSTM_128 (Bidirectional)      │ (None, 10, 512)        │     2,623,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Drop_2 (Dropout)                │ (None, 10, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ BiLSTM_64 (Bidirectional)       │ (None, 256)            │       656,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Drop_3 (Dropout)                │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pit_probability (Dense)         │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,754,113 (21.95 MB)

 Trainable params: 5,754,113 (21.95 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 118s 111ms/step - accuracy: 0.7567 - loss: 0.8520 - val_accuracy: 0.7842 - val_loss: 0.4318 - learning_rate: 1.0000e-04
Epoch 2/50
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 168s 167ms/step - accuracy: 0.8936 - loss: 0.4069 - val_accuracy: 0.8435 - val_loss: 0.3788 - learning_rate: 1.0000e-04
Epoch 3/50
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 170s 169ms/step - accuracy: 0.9350 - loss: 0.2668 - val_accuracy: 0.8868 - val_loss: 0.3101 - learning_rate: 1.0000e-04
Epoch 4/50
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 172s 170ms/step - accuracy: 0.9538 - loss: 0.1950 - val_accuracy: 0.8991 - val_loss: 0.3382 - learning_rate: 1.0000e-04
Epoch 5/50
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 172s 170ms/step - accuracy: 0.9620 - loss: 0.1632 - val_accuracy: 0.9030 - val_loss: 0.3292 - learning_rate: 1.0000e-04
Epoch 6/50
1008/1008 ━━━━━━━━━━━━━━━━━━━━ 173s 171ms/step - accuracy: 0.9677 - loss: 0.1401 - val_accuracy: 0.9062 - val_loss: 0.2972 - learning_rate: 1.0000e-04
Epoch 7/50
1008/1008 ━━━━━━━